calling openai endpoint without using the openai package

In [29]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - Please check the troubleshooting steps!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [30]:
import requests 

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}\

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}
    ]
}
payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [31]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-EYdH7ZYdkgNvCNMQIEeh1r1wgdqVa',
 'object': 'chat.completion',
 'created': 1791922833,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: Bananas are berries, but strawberries aren’t. Botanically, a berry comes from a single ovary of one flower and contains seeds inside; bananas fit that rule, while strawberries are actually aggregate fruits with seeds on the outside. Want another fun fact?',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 767,
  'total_tokens': 778,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 704,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [32]:
response.json()["choices"][0]["message"]["content"]

'Fun fact: Bananas are berries, but strawberries aren’t. Botanically, a berry comes from a single ovary of one flower and contains seeds inside; bananas fit that rule, while strawberries are actually aggregate fruits with seeds on the outside. Want another fun fact?'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [33]:
# Create OpenAI client
from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact about human habit to limit our lives mentally!"}])
response.choices[0].message.content

'Fun fact: About 40–50% of the actions we take each day are habitual—performed automatically rather than chosen in the moment. Our brains love routines because they save mental energy, but that autopilot can keep us from trying new things or breaking out of limiting patterns.\n\nIf you want to nudge yourself beyond those mental limits:\n- Use implementation intentions: plan specific if-then scenarios (e.g., If I feel stuck on a task, I’ll write one sentence to get started).\n- Try habit stacking: attach a new, small habit to an existing one (e.g., after brushing teeth, do two minutes of stretching).\n- Reduce friction for good habits and increase friction for bad ones: make the desirable option easy to start, and the undesired option harder to continue.\n\nWant another fun fact or tips to shake up a specific habit?'

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [34]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [35]:
# The ! tells certain notebook environments, such as Jupyter Notebook or Google Colab, to execute the following text as a shell command rather than as Python code.
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 


In [36]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

response = ollama.chat.completions.create(model="llama3.2", messages=[{
    "role": "user",
    "content": "Tell me a fun fact about Shih Tzu's"
}])

response.choices[0].message.content

'Here\'s a fun fact about Shih Tzus:\n\nShih Tzus are called "lion dogs" in Chinese culture, not because they resemble lions, but because the breed\'s original name "Lion Dog" referred to the mythical creatures that surrounded the emperor in ancient China. These mythical creatures were believed to protect the emperor and were often depicted as a combination of a lion and a dog. The Shih Tzu\'s long, flowing coat and regal demeanor earned it a place in the hearts (and imaginations) of Chinese royalty and art lovers alike.'

# EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

In [37]:
# Solution

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [38]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [39]:
ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
F

In [40]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [41]:
def messages_for(website):
   return [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_prefix + website}
    ]

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula

In [42]:
def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [43]:
summarize("https://edwarddonner.com")

'# Summary\nA website about artificial intelligence (AI) and its intersection with code, music, and more, founded by Ed Donner, a passionate AI enthusiast and expert. Ed shares his knowledge through Udemy courses and writes about AI-related topics on this site.\n\n# News and Announcements \n- Recent articles on resources for AI Coder, Builder with n8n, AI Engineering MLOps Track, and an optimal order for taking AI courses.'

In [44]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [45]:
display_summary("https://anthropic.com")

### Website Summary

Anthropic is a public benefit corporation aiming to harness the benefits of AI while minimizing its risks. The site's home page dives straight into research topics, including alignment, economics, and frontier red teams, hinting at their focus on developing safe and responsible AI solutions.

### News/Announcements

The website recently released Haiku 5.5, a new low-cost, high-volume small model that promises to be their fastest, cheapest, and most capable small model yet. This release is a milestone in Anthropic's commitment to making AI more accessible and beneficial.